# 🎬 IMDB Sentiment Analysis — ML Pipeline
**Dataset:** [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)  
**Task:** Binary classification — `positive` / `negative`  
**Pipeline:** TF-IDF + Logistic Regression → saved model → Streamlit app

---

## Phase 1 — Imports & Setup

In [ ]:
import re
import os
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")
os.makedirs("../models", exist_ok=True)

print("All imports successful.")

## Phase 2 — Load Dataset
> **Column names are `review` and `sentiment`** — not `text`/`label`. IMDB is binary only (no Neutral class).

In [ ]:
# FIX: Correct filename for the Kaggle IMDB download
df = pd.read_csv("../data/IMDB Dataset.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nClass distribution:")
print(df["sentiment"].value_counts())

## Phase 3 — Data Cleaning & Preprocessing
Key fixes vs a naive approach:
- **HTML tags stripped first** — IMDB reviews contain `<br />`, `<p>` etc.
- HTML entities decoded (`&amp;`, `&#39;` etc.)
- Bigram-friendly: only removes non-alpha chars, keeps spacing clean

In [ ]:
def clean_text(text):
    text = str(text)
    # 1. Strip HTML tags — critical for IMDB
    text = re.sub(r"<[^>]+>", " ", text)
    # 2. Decode HTML entities
    text = (text.replace("&amp;", "&").replace("&lt;", "<")
                .replace("&gt;", ">").replace("&quot;", '"').replace("&#39;", "'"))
    # 3. Lowercase
    text = text.lower()
    # 4. Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # 5. Remove @mentions
    text = re.sub(r"@\w+", "", text)
    # 6. Remove hashtag symbol, keep word
    text = re.sub(r"#", "", text)
    # 7. Keep only alphabetic characters
    text = re.sub(r"[^a-z\s]", "", text)
    # 8. Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Drop nulls and duplicates before cleaning
df = df.drop_duplicates().dropna(subset=["review", "sentiment"])
df = df[df["review"].str.strip() != ""].reset_index(drop=True)

df["clean_text"] = df["review"].apply(clean_text)

# Sanity check — no HTML tags should survive
assert df["clean_text"].str.contains("<br").sum() == 0, "HTML tags still present!"
print("HTML tag check passed.")
print(f"\nSample cleaned review:\n{df['clean_text'].iloc(0)[0][:300]}")

## Phase 4 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Class distribution
df["sentiment"].value_counts().plot(
    kind="bar", ax=axes[0],
    color=["#2196F3", "#FF5722"], edgecolor="white", width=0.5
)
axes[0].set_title("Class distribution", fontsize=13)
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

# Review length by class
df["text_length"] = df["clean_text"].apply(len)
for sentiment, color in [("positive", "#2196F3"), ("negative", "#FF5722")]:
    subset = df[df["sentiment"] == sentiment]["text_length"]
    axes[1].hist(subset, bins=60, alpha=0.6, color=color, label=sentiment)
axes[1].set_title("Review length distribution", fontsize=13)
axes[1].set_xlabel("Character count")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nMean review length by sentiment:")
print(df.groupby("sentiment")["text_length"].mean().round(1))

In [ ]:
# Per-class top-word frequency — filters generic movie terms
STOPWORDS = {
    "film", "movie", "the", "a", "an", "is", "it", "in", "of", "and", "to",
    "was", "that", "this", "i", "for", "with", "he", "she", "they", "on",
    "at", "be", "as", "but", "not", "have", "had", "his", "her", "one",
    "its", "are", "were", "by", "from", "so", "there", "br", "also", "just",
    "do", "if", "my", "me", "we", "or", "all", "would", "been", "which",
    "their", "about", "who", "out", "up", "can", "more", "no", "what", "when"
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, sentiment, color in zip(axes, ["positive", "negative"], ["#2196F3", "#FF5722"]):
    text = " ".join(df[df["sentiment"] == sentiment]["clean_text"])
    words = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    common = Counter(words).most_common(20)
    labels, counts = zip(*common)
    ax.barh(labels[::-1], counts[::-1], color=color, edgecolor="white")
    ax.set_title(f"Top 20 words — {sentiment} reviews", fontsize=13)
    ax.set_xlabel("Frequency")

plt.tight_layout()
plt.show()

## Phase 5 — Label Encoding
Encode string labels to integers. Save the encoder — needed to decode predictions back to `"positive"` / `"negative"` in `app.py`.

In [ ]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["sentiment"])

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
print("Label distribution:")
print(df["label"].value_counts())

joblib.dump(le, "../models/label_encoder.pkl")
print("\nlabel_encoder.pkl saved.")

## Phase 6 — Train / Test Split

In [ ]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size : {len(X_train):,}")
print(f"Test size  : {len(X_test):,}")
print(f"\nTrain class balance:\n{pd.Series(y_train).value_counts()}")

## Phase 7 — Build ML Pipeline
TF-IDF parameters tuned for IMDB:
- `min_df=3` — drops rare noise tokens
- `max_df=0.90` — drops near-universal terms like "film", "movie"  
- `ngram_range=(1,2)` — captures "not good", "highly recommend"
- `sublinear_tf=True` — log-normalises term frequency (handles long reviews)

In [ ]:
model_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=10000,
        min_df=3,
        max_df=0.90,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        C=1.0,
        solver="lbfgs",
        class_weight="balanced"
    ))
])

print("Pipeline created:")
print(model_pipeline)

## Phase 8 — Train the Model

In [ ]:
print("Training...")
model_pipeline.fit(X_train, y_train)
print("Done.")

## Phase 9 — Evaluate

In [ ]:
y_pred = model_pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Confidence distribution
probs = model_pipeline.predict_proba(X_test).max(axis=1)

plt.figure(figsize=(7, 4))
plt.hist(probs, bins=50, color="#2196F3", edgecolor="white", alpha=0.85)
plt.axvline(probs.mean(), color="#FF5722", linestyle="--", label=f"Mean: {probs.mean():.2f}")
plt.title("Prediction confidence distribution", fontsize=13)
plt.xlabel("Confidence score")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean confidence : {probs.mean():.3f}")
print(f"Low confidence (<0.6) predictions: {(probs < 0.6).sum()}")

## Phase 10 — Save Model

In [ ]:
import sklearn

MODEL_PATH = "../models/sentiment_model.pkl"
joblib.dump(model_pipeline, MODEL_PATH)
print(f"Model saved      : {MODEL_PATH}")

joblib.dump(le, "../models/label_encoder.pkl")
print(f"Encoder saved    : ../models/label_encoder.pkl")

# Log sklearn version — pickle files are version-sensitive
with open("../models/sklearn_version.txt", "w") as f:
    f.write(sklearn.__version__)
print(f"sklearn version  : {sklearn.__version__} (logged)")

## Phase 11 — Test Prediction Function
Verify the full predict flow: raw text → `clean_text()` → model → decoded label.  
> **Important:** `clean_text()` must always be applied before `model.predict()`. The model was trained on cleaned text.

In [ ]:
def predict_sentiment(raw_text: str) -> tuple:
    """
    Predict sentiment from raw (uncleaned) input text.
    Returns: (label: str, confidence: float)
    """
    if not raw_text or not raw_text.strip():
        raise ValueError("Input text cannot be empty.")
    cleaned = clean_text(raw_text)
    pred_int = model_pipeline.predict([cleaned])[0]
    confidence = float(model_pipeline.predict_proba([cleaned]).max())
    label = le.inverse_transform([pred_int])[0]
    return label, confidence

# Test cases
test_inputs = [
    "This movie was absolutely brilliant. The acting was superb!",
    "Terrible film. Waste of two hours. The plot made no sense at all.",
    "The cinematography was stunning but the story felt hollow.<br />",   # HTML in input
    "An average film, nothing special but not unwatchable either.",
]

print(f"{'Review':<65} {'Sentiment':<12} {'Confidence'}")
print("-" * 90)
for text in test_inputs:
    label, conf = predict_sentiment(text)
    print(f"{text[:63]:<65} {label:<12} {conf:.2%}")

## Phase 12 — Word Clouds (Optional)
Run this cell only if `wordcloud` is installed: `pip install wordcloud`

In [ ]:
try:
    from wordcloud import WordCloud

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, sentiment, bg in zip(axes, ["positive", "negative"], ["#e8f5e9", "#fce4ec"]):
        text = " ".join(df[df["sentiment"] == sentiment]["clean_text"])
        wc = WordCloud(
            width=700, height=350,
            background_color="white",
            colormap="RdYlGn" if sentiment == "positive" else "RdYlBu_r",
            max_words=150
        ).generate(text)
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
        ax.set_title(f"Word cloud — {sentiment}", fontsize=13)

    plt.tight_layout()
    plt.show()

except ImportError:
    print("wordcloud not installed. Run: pip install wordcloud")

## ✅ Summary

| Step | Detail |
|---|---|
| Dataset | IMDB 50K reviews — columns `review`, `sentiment` |
| Preprocessing | HTML-aware `clean_text()` — strips `<br />`, entities, lowercases |
| Label encoding | `LabelEncoder` saved as `label_encoder.pkl` |
| Vectorisation | TF-IDF: `max_features=10000`, `ngram_range=(1,2)`, `min_df=3`, `max_df=0.90` |
| Classifier | Logistic Regression (`class_weight="balanced"`, `max_iter=1000`) |
| Expected accuracy | ~89–92% on test set |
| Saved files | `models/sentiment_model.pkl`, `models/label_encoder.pkl`, `models/sklearn_version.txt` |

**Next step:** Run `streamlit run app.py` to launch the web interface.